In [14]:

class PuzzleState:
    def __init__(self, board, blank_row, blank_col, depth=0, parent=None, move="Initial"):
        self.board = board
        self.blank_row = blank_row
        self.blank_col = blank_col
        self.depth = depth
        self.parent = parent
        self.move = move

    def get_tuple(self):
        return tuple(tuple(row) for row in self.board)

def print_board(board):
    for row in board:
        print("  " + " ".join(map(str, row)))
    print("  " + "-" * 7)

def get_neighbors(state):
    neighbors = []
    # Movements defined as: (row_offset, col_offset, move_name)
    moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

    for dr, dc, move_name in moves:
        new_r, new_c = state.blank_row + dr, state.blank_col + dc

        if 0 <= new_r < 3 and 0 <= new_c < 3:
            new_board = [row[:] for row in state.board]
            # Swap empty space (0) with target tile
            new_board[state.blank_row][state.blank_col], new_board[new_r][new_c] = \
                new_board[new_r][new_c], new_board[state.blank_row][state.blank_col]

            neighbors.append(PuzzleState(new_board, new_r, new_c, state.depth + 1, state, move_name))

    return neighbors

def solve_and_trace_dfs(start_board, goal_board, max_depth=15):
    # Locate the initial 0 coordinate
    start_r, start_col = 0, 0
    for r in range(3):
        for c in range(3):
            if start_board[r][c] == 0:
                start_r, start_col = r, c
                break

    start_state = PuzzleState(start_board, start_r, start_col)
    goal_tuple = tuple(tuple(row) for row in goal_board)

    stack = [start_state]
    visited = {}
    iteration = 0

    print("=" * 45)
    print("🔄 STARTING STATE EXPLORATION TRACE 🔄")
    print("=" * 45)

    while stack:
        current = stack.pop()
        iteration += 1

        # --- TRACE LOGGING FOR EVERY VISITED STATE ---
        print(f"\n[Iteration #{iteration}] Evaluated State Details:")
        print(f"  • Last Action: {current.move}")
        print(f"  • Current Branch Depth: {current.depth}")
        print("  • Board Layout:")
        print_board(current.board)
        # ---------------------------------------------

        if current.get_tuple() == goal_tuple:
            print(f"🎯 GOAL STATE MATCH FOUND AT ITERATION #{iteration}!")
            return current

        if current.depth >= max_depth:
            print("  ⚠️ Depth limit reached on this branch. Backtracking...")
            continue

        if current.get_tuple() not in visited or current.depth < visited[current.get_tuple()]:
            visited[current.get_tuple()] = current.depth

            # Add valid subsequent moves onto the processing Stack
            for neighbor in get_neighbors(current):
                stack.append(neighbor)

    return None

def print_final_solution(solution_state):
    path = []
    curr = solution_state
    while curr:
        path.append(curr)
        curr = curr.parent
    path.reverse()

    print("\n" + "=" * 45)
    print(f"🏆 SUCCESS! Final Solution Path established in {len(path) - 1} moves:")
    print("=" * 45)
    for step, state in enumerate(path):
        print(f"Step {step}: Move Taken -> {state.move} (Depth: {state.depth})")
        print_board(state.board)

# --- TRACE CONFIGURATION RUN ---
if __name__ == "__main__":
    # Keeping the setup closely bound to show a clean, fast trace output sequence
    # '0' represents the empty tile space
    initial_configuration = [
        [2,8,3],
        [1,6,4],
        [7,0,5]
    ]

    goal_configuration = [
        [1,2,3],
        [8,0,4],
        [7,6,5]
    ]

    # max_depth threshold prevents memory overflow from un-demarcated recursion
    result = solve_and_trace_dfs(initial_configuration, goal_configuration, max_depth=10)

    if result:
        print_final_solution(result)
    else:
        print("\n❌ System timed out or depth boundaries cut off branches before hitting goal state.")


🔄 STARTING STATE EXPLORATION TRACE 🔄

[Iteration #1] Evaluated State Details:
  • Last Action: Initial
  • Current Branch Depth: 0
  • Board Layout:
  2 8 3
  1 6 4
  7 0 5
  -------

[Iteration #2] Evaluated State Details:
  • Last Action: Right
  • Current Branch Depth: 1
  • Board Layout:
  2 8 3
  1 6 4
  7 5 0
  -------

[Iteration #3] Evaluated State Details:
  • Last Action: Left
  • Current Branch Depth: 2
  • Board Layout:
  2 8 3
  1 6 4
  7 0 5
  -------

[Iteration #4] Evaluated State Details:
  • Last Action: Up
  • Current Branch Depth: 2
  • Board Layout:
  2 8 3
  1 6 0
  7 5 4
  -------

[Iteration #5] Evaluated State Details:
  • Last Action: Left
  • Current Branch Depth: 3
  • Board Layout:
  2 8 3
  1 0 6
  7 5 4
  -------

[Iteration #6] Evaluated State Details:
  • Last Action: Right
  • Current Branch Depth: 4
  • Board Layout:
  2 8 3
  1 6 0
  7 5 4
  -------

[Iteration #7] Evaluated State Details:
  • Last Action: Left
  • Current Branch Depth: 4
  • Board L

In [17]:
class PuzzleState:
    def __init__(self, board, parent=None, move="", depth=0):
        self.board = board        # 1D list of length 9 representing the 3x3 grid
        self.parent = parent      # Link to parent state to reconstruct the path
        self.move = move          # The move that led to this state
        self.depth = depth        # Current depth of the node

    def __eq__(self, other):
        # FIX: Check if the compared object is a PuzzleState instance
        if not isinstance(other, PuzzleState):
            return False
        return self.board == other.board

    def __hash__(self):
        return hash(tuple(self.board))

    def find_blank(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank_idx = self.find_blank()
        row, col = blank_idx // 3, blank_idx % 3

        moves = {
            "Up": (-1, 0, blank_idx - 3),
            "Down": (1, 0, blank_idx + 3),
            "Left": (0, -1, blank_idx - 1),
            "Right": (0, 1, blank_idx + 1)
        }

        for move_name, (dr, dc, new_idx) in moves.items():
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_board = list(self.board)
                new_board[blank_idx], new_board[new_idx] = new_board[new_idx], new_board[blank_idx]
                neighbors.append(PuzzleState(new_board, self, move_name, self.depth + 1))

        return neighbors

def print_board(board):
    for i in range(0, 9, 3):
        print(f" {board[i]} {board[i+1]} {board[i+2]} ")
    print("-" * 11)

def reconstruct_path(state):
    path = []
    current = state
    while current.parent is not None:
        path.append((current.move, current.board))
        current = current.parent
    return path[::-1]

def depth_limited_search(state, goal_board, limit, visited_path):
    if state.board == goal_board:
        return state

    if state.depth >= limit:
        return "LIMIT"

    any_remaining = False
    visited_path.add(tuple(state.board))

    for neighbor in state.get_neighbors():
        if tuple(neighbor.board) not in visited_path:
            result = depth_limited_search(neighbor, goal_board, limit, visited_path)
            if result == "LIMIT":
                any_remaining = True
            elif result is not None:
                return result

    visited_path.remove(tuple(state.board))
    return "LIMIT" if any_remaining else None

def iterative_deepening_search(initial_board, goal_board, max_depth=50):
    initial_state = PuzzleState(initial_board)

    for depth in range(max_depth):
        visited_path = set()
        result = depth_limited_search(initial_state, goal_board, depth, visited_path)

        if result != "LIMIT" and result is not None:
            return result

    return None

if __name__ == "__main__":
    # Formatted mapping for your custom layout inputs
    initial_configuration = [2, 8, 3, 1, 6, 4, 7, 0, 5]
    goal_configuration    = [1, 2, 3, 8, 0, 4, 7, 6, 5]

    print("Initial State:")
    print_board(initial_configuration)
    print("Goal State:")
    print_board(goal_configuration)

    print("Searching for solution via IDS...")
    solution_node = iterative_deepening_search(initial_configuration, goal_configuration)

    if solution_node:
        path = reconstruct_path(solution_node)
        print(f"\n🎉 Success! Goal reached at depth: {solution_node.depth}")
        print(f"Total steps required: {len(path)}\n")

        current_step = 1
        for move, board in path:
            print(f"Step {current_step}: Move blank space '{move}'")
            print_board(board)
            current_step += 1
    else:
        print("\n❌ Could not find a solution within the depth limits.")


Initial State:
 2 8 3 
 1 6 4 
 7 0 5 
-----------
Goal State:
 1 2 3 
 8 0 4 
 7 6 5 
-----------
Searching for solution via IDS...

🎉 Success! Goal reached at depth: 5
Total steps required: 5

Step 1: Move blank space 'Up'
 2 8 3 
 1 0 4 
 7 6 5 
-----------
Step 2: Move blank space 'Up'
 2 0 3 
 1 8 4 
 7 6 5 
-----------
Step 3: Move blank space 'Left'
 0 2 3 
 1 8 4 
 7 6 5 
-----------
Step 4: Move blank space 'Down'
 1 2 3 
 0 8 4 
 7 6 5 
-----------
Step 5: Move blank space 'Right'
 1 2 3 
 8 0 4 
 7 6 5 
-----------
